# Harness Engineering: OpenCode Case Study
## Chapter 3: Subagents and Delegation

In the previous labs, one agent read the project, changed the code, and ran the tests. This time, we will separate finding the problem from fixing it.

We will use the same grade-calculation bug. The **main agent** will ask an **investigator agent** to read the files, explain the cause, and suggest a change. The investigator agent will not edit files. The main agent will check the suggestion, make the change, and run the test.

Giving another agent a smaller task is called **delegation**. The agent receiving that task is a **subagent**.

First, run the repair with one agent. Then try this division of work. Follow the request, the returned findings, and the main agent's next actions to see whether the extra investigation was useful.

**Part 1** runs the experiment. **Part 2** explains what changes when you delegate. **Part 3** traces the same steps in OpenCode's source.


### Part 1: Ask Another Agent to Investigate
#### 1. Reproduce the bug

Run the setup in a fresh Colab runtime. It installs OpenCode `1.18.34` and copies Chapter 1's four-file example into a new folder. It does not call a model.

Read the requirements, program, data, and test below. Both runs will start with these same files. We leave out Chapter 2's report and `AGENTS.md` so the task is just to repair the calculation.


In [ ]:
#@title Prepare OpenCode and the example project
from pathlib import Path
import copy
import subprocess
import sys
import tempfile

REPO_URL = "https://github.com/SleepyLGod/AIST4020-Lab-Tutorials.git"
LAB_RELATIVE = Path("labs/opencode/subagents")
LAB_DIR = next((p for p in (Path.cwd(), Path.cwd() / LAB_RELATIVE)
                if (p / "subagents_support.py").is_file()), None)
if LAB_DIR is None:
    course = Path(tempfile.mkdtemp(prefix="aist4020-course-"))
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(course)], check=True)
    LAB_DIR = course / LAB_RELATIVE
sys.path.insert(0, str(LAB_DIR))
import subagents_support as support
if Path(support.__file__).resolve() != (LAB_DIR / "subagents_support.py").resolve():
    raise RuntimeError("Restart the runtime to load this lab's helper.")
lab = support.open_lab(version="1.18.34")


In [ ]:
support.show_project(lab.project)


Check the starting program yourself. For 80, 100, and a missing score, the average should be 90. The test should fail with `60.0 != 90.0`.


In [ ]:
TEST_COMMAND = ["python3", "-m", "unittest", "-v"]
initial_test = lab.check(TEST_COMMAND)
assert not initial_test.timed_out and initial_test.returncode != 0
assert "60.0 != 90.0" in initial_test.stdout + initial_test.stderr


**Choose a model.**

The default is `ollama`, with model calls enabled (`ENABLE_MODEL_CALLS = True`). Use a GPU runtime; no API key is needed. With Ollama selected, the next cell installs and starts Ollama and downloads the model; allow time for the first run. To use `deepseek` or `nvidia` instead, select it below and enter your key when prompted, or save it in Colab Secrets.

`base_url` is the model service address, and `model` selects the model. `key_env` names the environment variable that holds your key; do not replace it with the key itself. To use your own provider, fill in the `custom` entry and select it.

For NVIDIA, this example selects [GLM-5.3-Flash](https://build.nvidia.com/z-ai/glm-5-3-flash). You can browse the [NVIDIA model catalog](https://build.nvidia.com/models) for alternatives that support tool calling; copy the model ID from the model page's API example.

Keep the same model for both runs and both roles. API calls may use paid credit, including the investigator's calls.


In [ ]:
BACKENDS = {
    "deepseek": dict(base_url="https://api.deepseek.com/v1",
                     model="deepseek-flash", key_env="DEEPSEEK_API_KEY"),
    "nvidia": dict(base_url="https://integrate.api.nvidia.com/v1",
                   model="z-ai/glm-5.3-flash", key_env="NVIDIA_NIM_API_KEY"),
    "ollama": dict(base_url="http://127.0.0.1:11434/v1",
                   model="qwen3.5:9b-q4_K_M", key_env=""),
    "custom": dict(base_url="", model="", key_env=""),
}

backend = "ollama"
ENABLE_MODEL_CALLS = True
lab.configure(backend, BACKENDS, enabled=ENABLE_MODEL_CALLS)


#### 2. Let one agent do the whole repair

First, ask the main agent to investigate and fix the bug on its own. It can read and search files, edit only `grades.py`, and run our test command. OpenCode uses the `task` tool to request work from another agent; we deny that tool for this run.

We use OpenCode's existing `build` agent and give it two instructions:

- `MAIN_INSTRUCTIONS` tells it how to work. It goes into the agent's `prompt`, replacing its default model instructions.
- `REPAIR_TASK` is the separate request telling it what to fix this time.

`permission` controls its tools. `steps` tells OpenCode when to ask the model to finish; it does not guarantee an exact number of actions.


In [ ]:
MAIN_PERMISSIONS = {
    "*": "deny",
    "read": "allow", "glob": "allow", "grep": "allow",
    "edit": {"*": "deny", "grades.py": "allow", "**/grades.py": "allow"},
    "bash": {"*": "deny", "python3 -m unittest -v": "allow"},
    "task": "deny",
    "external_directory": "deny",
}
MAIN_INSTRUCTIONS = (
    "Use read, glob, and grep to inspect project files. "
    "Use bash only for python3 -m unittest -v. "
    "Check any advice against the files before editing. "
    "Report only tests you actually ran."
)
SINGLE_AGENTS = {
    "build": {"prompt": MAIN_INSTRUCTIONS, "steps": 12,
              "permission": MAIN_PERMISSIONS},
}
REPAIR_TASK = """Fix the average calculation to match README.md.
Run python3 -m unittest -v before changing the implementation and again after
the last change. Only modify grades.py.
Briefly report the cause, change, and test results you actually observed."""

SINGLE_SETTINGS = {"permission": {"*": "deny"}, "agent": SINGLE_AGENTS}
single = lab.new_case("One agent", SINGLE_SETTINGS)


`SINGLE_SETTINGS["agent"]["build"]` is the role definition OpenCode reads. `lab.new_case(...)` saves this configuration with the model connection and makes a fresh project. The printed configuration path is passed through `OPENCODE_CONFIG`; the command below runs in `single.project`.

Run the request below. The notebook prints the command and records OpenCode's output; the model chooses which tools to request. The notebook stops a run after five minutes without retrying. To retry, read the error, rerun the cell that creates `single` or `delegated`, then rerun its task.


In [ ]:
RUN_COMMAND = [str(lab.binary), "run", "--pure", "--format", "json",
               "--agent", "build", "--model", lab.model]
single_result = single.run([*RUN_COMMAND, "Investigate the project yourself. " + REPAIR_TASK])


Check what happened before adding another agent. The next cell runs your test, loads the saved conversation, and displays the actions and file changes. If the conversation cannot be loaded, it shows the available output and file checks and marks the missing record.

Did the main agent edit `grades.py` and run the test afterward? `check(...)` runs directly from the notebook, outside the agent's tool permissions. Your test checks the result; it does not count as the agent retesting.


In [ ]:
single_test = single.check(TEST_COMMAND)
single.load_sessions()
support.show_case(single, single_test)


#### 3. Give the investigator agent the investigation

Now split the work. Define an agent named `investigator` to identify the cause, suggest a change, and say what to test. It can read and search files, but cannot edit, run commands, or delegate to another agent. The main agent keeps the repair and testing job.

We copy the first run's settings, add `investigator`, and allow the main agent to request it through `task`. The investigator agent's own `task` permission stays denied.

`description` tells the main agent what the investigator agent can do. `INVESTIGATOR["prompt"]` tells the investigator agent how to investigate and report its findings. `mode="subagent"` lets the main agent delegate a task to it.

`lab.model` is the model you selected earlier, also used by the main agent. For example, `course/deepseek-flash` means `deepseek-flash` through our connection named `course`. Both roles use the same model so we can focus on the division of work.

`delegated` starts from a new copy of the original buggy project, separate from `single`. Within this second run, the main agent and investigator agent share that project folder, but keep separate conversations.


In [ ]:
INVESTIGATOR = {
    "description": "Read-only investigator agent: locate a grade-calculation bug and return advice with file evidence.",
    "mode": "subagent",
    "model": lab.model,
    "steps": 6,
    "prompt": """Read the project files named in your task and return advice.
Do not modify files, run commands, or delegate.
Keep the final reply under 120 words, with three bullets:
- Cause: identify the mismatch and cite the file and line.
- Suggested change: describe the smallest repair.
- Check: say what the caller should test. Do not claim you ran it.
Do not repeat whole files or provide a full replacement program.""",
    "permission": {
        "*": "deny",
        "read": "allow", "glob": "allow", "grep": "allow",
        "edit": "deny", "bash": "deny", "task": "deny",
        "external_directory": "deny",
    },
}

DELEGATED_SETTINGS = copy.deepcopy(SINGLE_SETTINGS)
DELEGATED_SETTINGS["agent"]["investigator"] = INVESTIGATOR
DELEGATED_SETTINGS["agent"]["build"]["permission"]["task"] = {"*": "deny", "investigator": "allow"}

delegated = lab.new_case("With investigator agent", DELEGATED_SETTINGS)
assert single.before == delegated.before  # Both runs start with the same files.
investigator_config = delegated.inspect_agent("investigator")


The table above comes from `opencode debug agent investigator`, not from a model reply. Check that only reading and searching are enabled. The `edit` permission also controls the `write` tool, so a different write tool does not give the investigator agent permission to change files.

Send `delegation_request` below to the **main agent**. It asks the main agent to get a diagnosis before making the repair. If the main agent calls `task`, it writes a separate `prompt` for the investigator agent. Next, check what it actually sent and what came back.


In [ ]:
delegation_request = """First use the task tool to ask investigator to read
README.md, grades.py, test_grades.py, and scores.csv.
Ask for a reply under 120 words: the cause with file/line evidence,
the smallest suggested change, and what you should test.
The investigator must not modify files or run commands.
After its reply, verify the advice against the files and complete the repair.
""" + REPAIR_TASK

delegated_result = delegated.run([*RUN_COMMAND, delegation_request])


#### 4. Follow the request and the returned advice

Compare your `delegation_request` above with the main agent's actual `task` request below: what files and questions did it give the investigator agent? Then read the returned advice. Does it identify a cause and point to code you can check?

To see how the investigator agent reached its answer, inspect its **child session**, a separate saved conversation containing its file reads. The notebook loads it using the child session ID recorded by OpenCode.


In [ ]:
delegated.load_sessions()
support.show_handoffs(delegated)


The child session's `parentID` links it to the main session. Intermediate file contents stay in the child's records unless the investigator agent includes them in its reply.

If the investigation completed, find the advice under **What the main agent received**. Then follow the main agent's actions: did it check the cited code, make a change, and run the test? If the run stopped earlier, use the status and last action or error to locate where it stopped. Progress text is not a finished investigation; missing records leave that part unknown.

If no `task` call appears, this run did not demonstrate delegation. You can retry with a fresh case, or trace the child session and returned reply in Part 3.

The test below checks the program either way.


In [ ]:
delegated_test = delegated.check(TEST_COMMAND)
support.show_case(delegated, delegated_test)


#### 5. Did the division of work help?

Now compare the two runs. They started with the same program, model, and repair goal. Check the files and test results first, then compare how the work was done. Include the investigator agent's actions, shown separately in the table.

Did the advice help the main agent find and fix the problem, or did it investigate everything again? Rereading the cited lines is useful verification; repeating the whole search may save little work. If either run did not finish, compare where it stopped rather than assuming delegation helped or hurt. One pair of runs cannot establish which arrangement is generally faster or better.


In [ ]:
support.compare_cases(single, delegated,
                      single_test=single_test, delegated_test=delegated_test)


### Part 2: What Delegation Changes
#### 1. A task becomes two connected conversations

In the first run, the main agent works in one conversation. In the second, it can ask the investigator agent to investigate in a separate conversation and return its findings. Both use the same model; their jobs differ because we give them different instructions and permissions.

Our request names four files and asks for a cause, suggested change, and test recommendation. The read-only rules keep the investigator agent focused on investigation. The main agent still has to check the advice and complete the repair.

![Example workflow: the main agent sends an investigation task, the investigator agent reads the project and returns advice, then the main agent verifies, edits, and tests. Both access the same project files.](https://raw.githubusercontent.com/SleepyLGod/AIST4020-Lab-Tutorials/main/labs/opencode/subagents/assets/delegation.png)

*This diagram shows the intended workflow. The tables in Part 1 show what your run actually did.*


#### 2. Separate history, shared files

The investigator agent has its own conversation. In this example, OpenCode starts it with the delegated request and the investigator agent's instructions, rather than copying the main conversation's entire history.

Both agents still access the **same project folder**. A separate session is not a separate filesystem. Our investigator agent is read-only because of its tool permissions, not because it is a subagent.

The returned findings become part of the main agent's conversation. The main agent does not need every line the investigator agent read, but it does need enough evidence to check the recommendation.

| What you inspected | What it tells you |
| --- | --- |
| The main agent's `task` input | What the investigator agent was asked to investigate |
| The child session's `parentID` | Which main session delegated the task |
| Investigator agent's actions and reply | What it inspected and what it concluded |
| Main agent's actions after the reply | Whether it checked the advice and completed the repair |
| Your test and file comparison | Whether the result meets the example's requirements |


#### 3. When is a subagent useful?

Our small bug needs little investigation. One agent can handle it, and delegation may add repeated reading and a longer wait.

A subagent becomes more useful when it can do a well-defined piece of work and return something compact: for example, search several modules for callers of a function and return the locations that a repair may affect. The detailed search stays in its conversation; the main agent receives the findings it needs.

Before delegating, ask: **What result do I need back, and can I check it?** "Investigate why missing grades change the average; cite the relevant line" gives the subagent a clearer job than "help with this project."


#### The same idea in Claude Code

Claude Code also offers **subagents** with their own instructions, tools, and context. Its documentation includes a code reviewer example. The familiar pattern is to give a subagent a focused job and use its returned findings. [Claude Code subagents](https://code.claude.com/docs/en/sub-agents)

**Agent teams** add coordination among multiple Claude Code instances, including a shared task list and messages between members. They can be useful when members need to exchange findings and coordinate their work, rather than just return one result. The official documentation currently marks teams as experimental and disabled by default. [Claude Code agent teams](https://code.claude.com/docs/en/agent-teams)

For this grade repair, an investigator agent returning advice is enough. A team would add coordination without giving us a clearer experiment. The useful choice is about the work and communication needed, not the number of agents.


### Part 3: How OpenCode Runs the Subagent

Part 1 followed the investigation request and returned advice. Now trace how OpenCode connects those steps: it loads the role settings, checks the `task` request, runs the investigator agent in a child session, and returns the findings to the main agent.

The excerpts use [OpenCode v1.18.34](https://github.com/anomalyco/opencode/tree/v1.18.34). `// ...` omits unrelated lines. As in Chapter 2, read `yield*` as "perform this operation and get its result."


#### 1. Where does the investigator agent's definition go?

In Part 1, you put the investigator agent's job in `prompt` and its read-only rules in `permission`. The saved configuration contains `agent.investigator`. OpenCode reads those settings here:

```ts
for (const [key, value] of Object.entries(cfg.agent ?? {})) {
  // ...
  let item = agents[key]
  // ...
  if (value.model) item.model = Provider.parseModel(value.model)
  // ...
  item.prompt = value.prompt ?? item.prompt
  item.description = value.description ?? item.description
  // ...
  item.mode = value.mode ?? item.mode
  // ...
  item.steps = value.steps ?? item.steps
  // ...
  item.permission = Permission.merge(item.permission, Permission.fromConfig(value.permission ?? {}))
}
```

*Source: [agent/agent.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/agent/agent.ts)*

For our investigator agent, `key` is `"investigator"` and `value` is the definition you wrote. `item` is OpenCode's agent configuration. The short-answer instruction goes into `prompt`; the read-only rules go into `permission`. Its instructions and permissions define the job, not its name.


#### 2. Who is allowed to delegate to whom?

The first run denied `task`; the second allowed it for `investigator`. When the main agent requests `task` with `subagent_type: "investigator"`, this code checks that permission and looks up the agent:

```ts
if (!ctx.extra?.bypassAgentCheck) {
  yield* ctx.ask({
    permission: id,
    patterns: [params.subagent_type],
    // ...
  })
}

const next = yield* agent.get(params.subagent_type)
```

*Source: [tool/task.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/tool/task.ts)*

Here `id` is `"task"`, and `params.subagent_type` names the requested subagent. For our call, follow `ctx.ask`: it checks whether the main agent may delegate to `investigator`. If allowed, `agent.get(...)` loads the investigator agent's configuration into `next`.

This is why the permission belongs in the main agent's settings: it controls whom that agent may ask for help.


#### 3. How does the investigator agent get its own conversation?

The separate file-reading records you looked for in Part 1 belong to a child session. For a new delegation, OpenCode creates it and records which main session requested the work:

```ts
const nextSession =
  session ??
  (yield* sessions.create({
    parentID: ctx.sessionID,
    title: params.description + ` (@${next.name} subagent)`,
    agent: next.name,
    // ...
  }))
```

*Source: [tool/task.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/tool/task.ts)*

`ctx.sessionID` identifies the main conversation. When no child session is supplied, `sessions.create(...)` creates one and records that parent ID. `nextSession.id` identifies the child.

OpenCode then turns the delegated text into message parts and runs the chosen agent in that child session:

```ts
const runTask = Effect.fn("TaskTool.runTask")(function* () {
  const parts = yield* ops.resolvePromptParts(params.prompt)
  const result = yield* ops.prompt({
    messageID: MessageID.ascending(),
    sessionID: nextSession.id,
    model: {
      modelID: model.modelID,
      providerID: model.providerID,
    },
    // ...
    agent: next.name,
    parts,
  })
  // ...
  return result.parts.findLast((item) => item.type === "text")?.text ?? ""
})
```

*Source: [tool/task.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/tool/task.ts)*

`params.prompt` is the message the main agent wrote to the investigator agent. `parts` carries that message, `model` selects the model, and `agent: next.name` selects the investigator agent's instructions and tools.

The final line selects the last text part of the reply. It does not return the child's whole conversation.


#### 4. Why doesn't the investigator agent get the main agent's editing tools?

We allowed the main agent to edit `grades.py`, but asked the investigator agent only to investigate. OpenCode checks each tool request against the requesting agent's rules together with its session rules:

```ts
ask: (req) =>
  permission
    .ask({
      ...req,
      sessionID: input.session.id,
      tool: { messageID: input.processor.message.id, callID: options.toolCallId },
      ruleset: Permission.merge(input.agent.permission, input.session.permission ?? []),
    })
    .pipe(Effect.orDie),
```

*Source: [session/tools.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/tools.ts)*

`input.agent` is the agent making this request: the investigator agent for its file reads, the main agent for its edit. `req` describes the requested action. The message and call IDs identify which request is being checked.

The investigator agent's rules deny edits and commands. The main agent's permission to edit `grades.py` is not a permission grant to the investigator agent. Separate role settings let us limit each agent to the actions its job needs.

*Further reading: [How selected parent-session restrictions carry into a child](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/agent/subagent-permissions.ts).*


#### 5. How does the advice get back to the main agent?

The main agent waits for the investigator agent. When the investigation completes, the `task` tool returns its findings through this code:

```ts
return {
  title: params.description,
  metadata,
  output: renderOutput({ sessionID: nextSession.id, state: "completed", text: result?.output ?? "" }),
}
```

*Source: [tool/task.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/tool/task.ts)*

These snippets use two different variables named `result`. Inside `runTask`, it holds the investigator agent's response; `result.parts` lets OpenCode extract the reply text. Here, it holds the finished task's record, with that text in `result.output`.

`renderOutput(...)` wraps the text with the child session ID and completion state. If your investigation completed, look for this result under **What the main agent received** in Part 1.

From here, the same tool-result loop you traced in Chapter 1 continues: OpenCode saves the result and includes it in the main agent's next model input. The main agent can use the advice to choose its next action. Our example asks it to verify the recommendation, then edit and test.

Delegation changes who investigates and what returns to the main conversation. It does not remove the need to check the final repair.


To keep your run records, download them before disconnecting and deleting the Colab runtime. Review them before sharing.


In [ ]:
DOWNLOAD_RECORDS = False
if DOWNLOAD_RECORDS:
    lab.download_records()
